In [ ]:
import torch
import torch.nn.functional as F
import iceaggr
from iceaggr.models import FlatTransformerV2
print(iceaggr.__file__)        # should point into your work clone

torch.manual_seed(0)
config = dict(
    max_pulses_per_dom=41, d_model=256, max_doms=128, num_heads=8, num_layers=2,
    hidden_dim=512, head_hidden_dim=128, dropout=0.0, input_mode='none',
    head_type='pid', num_classes=3,
)
model = FlatTransformerV2(config)
model.train()

B = 32
dom_vectors = torch.randn(B, 128, 4 + 3 * 41)       # (32, 128, 127)
padding_mask = torch.ones(B, 128, dtype=torch.bool)
targets = torch.randint(0, 3, (B,))                 # 0 noise, 1 muon, 2 neutrino

out = model(dom_vectors, padding_mask)
print("logits:", out["logits"].shape, "| embedding:", out["embedding"].shape)

opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
for step in range(200):
    loss = F.cross_entropy(model(dom_vectors, padding_mask)["logits"], targets)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 40 == 0:
        print(step, round(loss.item(), 4))

In [3]:
import torch
from iceaggr.models.pid_metrics import confusion_matrix, metrics_from_confusion

targets = torch.tensor([0, 0, 1, 1, 2, 2])
logits = torch.tensor([[2., 0, 0], [0, 1., 0], [0, 3., 0], [0, 2., 0], [0, 0, 1.], [0, 0, 5.]])

cm = confusion_matrix(logits, targets)
print(cm)
print(metrics_from_confusion(cm))

tensor([[1, 1, 0],
        [0, 2, 0],
        [0, 0, 2]])
{'accuracy': 0.8333333333333334, 'per_class_accuracy': [0.5, 1.0, 1.0]}
